# components.helpers

> Shared rendering helpers for the workflow session management interface.

Most of these are lifted from `cjm-transcript-workflow-management/components/helpers.py` — they're workflow-agnostic and reusable as-is. One is new to this library:

- `render_active_session_badge` — visual indicator for the currently active session

In the longer term these workflow-agnostic helpers may be extracted into a shared `cjm-fasthtml-management-chrome` library so both management libraries import from one source. Out of scope for now.

In [ ]:
#| default_exp components.helpers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List, Optional

from fasthtml.common import Div, Span, H3, Button, P, Input

# DaisyUI components
# btn / btn_styles / btn_colors / btn_sizes still used by `render_icon_button`
# (a row-action helper that pre-dates V1 and represents a V1 catalog gap for
# row-action icon buttons). Other sites in this file use V1 buttons directly.
from cjm_fasthtml_daisyui.components.actions.button import (
    btn, btn_colors, btn_styles, btn_sizes
)
from cjm_fasthtml_daisyui.components.data_display.badge import (
    badge, badge_colors, badge_styles, badge_sizes
)
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

from cjm_fasthtml_design_system.text_tiers import text_tiers

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, text_align
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, gap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V1 button roles, V11 icon-size roles)
from cjm_fasthtml_design_system.buttons import buttons
from cjm_fasthtml_design_system.icons import icons

## Debug Flag

In [ ]:
#| export
DEBUG_SESSION_RENDER = False # Enable for verbose component rendering logs

## Section Header

A standard section header with a lucide icon and a title. Lifted verbatim from `cjm-transcript-workflow-management`.

In [ ]:
#| export
def render_section_header(
    title:str, # Section title text
    icon_name:str, # Lucide icon name (kebab-case)
) -> Any: # Header element with icon and title
    """Render a section header with a leading lucide icon."""
    return H3(
        lucide_icon(icon_name, size=icons.section_header),
        title,
        cls=combine_classes(
            font_size.lg, font_weight.semibold,
            flex_display, items.center, gap(2)
        )
    )

In [ ]:
header = render_section_header("Sessions", "layers")
assert header.tag == "h3"
# The icon is the first child.
assert header.children[0].tag == "svg"
# The title is the second child.
assert header.children[1] == "Sessions"
print(f"Header classes: {header.attrs.get('class')}")

Header classes: text-lg font-semibold flex items-center gap-2


## Icon Button

A ghost-style button with an icon and accessible tooltip label. Additional kwargs (e.g. `hx_post`, `onclick`) pass through to the underlying `<button>`.

In [ ]:
#| export
def render_icon_button(
    icon_name:str, # Lucide icon name (kebab-case)
    label:str, # Accessible tooltip label
    color:Optional[str]=None, # DaisyUI button color class (e.g. btn_colors.error)
    size:Optional[str]=None, # DaisyUI button size class (e.g. btn_sizes.sm)
    **kwargs # Additional HTML attributes (hx_post, onclick, etc.)
) -> Any: # Button element with icon
    """Render a ghost-style button with an icon and accessible label.
    
    Defaults to `type="button"` to prevent accidental form submission when
    rendered inside a wrapping form. Callers can override by passing `type=...`.
    """
    classes = [btn, btn_styles.ghost]
    if color: classes.append(color)
    if size: classes.append(size)
    kwargs.setdefault("type", "button")
    return Button(
        lucide_icon(icon_name, size=icons.icon_button),
        title=label,
        cls=combine_classes(*classes),
        **kwargs
    )

In [ ]:
delete_btn = render_icon_button("trash-2", "Delete session", color=btn_colors.error, size=btn_sizes.sm, hx_post="/delete")
assert delete_btn.tag == "button"
assert delete_btn.attrs["title"] == "Delete session"
assert delete_btn.attrs["type"] == "button"  # Prevent accidental form submission
assert "btn-error" in delete_btn.attrs["class"]
assert "btn-sm" in delete_btn.attrs["class"]
assert "btn-ghost" in delete_btn.attrs["class"]
assert delete_btn.attrs.get("hx-post") == "/delete"

# Callers can still override type if needed.
submit_btn = render_icon_button("check", "Submit", type="submit")
assert submit_btn.attrs["type"] == "submit"

print(f"Delete button classes: {delete_btn.attrs['class']}")

## Alert

A DaisyUI alert for inline feedback (errors, success, info). Use the `alert_colors` factory for semantic colors.

In [ ]:
#| export
def render_alert(
    message:str, # Alert message text
    color:Optional[str]=None, # DaisyUI alert color class (e.g. alert_colors.success)
    alert_id:str="", # Optional HTML ID for the alert
) -> Any: # Alert element
    """Render a DaisyUI alert message."""
    classes = [alert]
    if color: classes.append(color)
    kwargs = {"role": "alert", "cls": combine_classes(*classes)}
    if alert_id: kwargs["id"] = alert_id
    return Div(Span(message), **kwargs)

In [ ]:
err = render_alert("Session not found", color=alert_colors.error, alert_id="wsm-alert")
assert err.tag == "div"
assert err.attrs["role"] == "alert"
assert err.attrs["id"] == "wsm-alert"
assert "alert" in err.attrs["class"]
assert "alert-error" in err.attrs["class"]
assert err.children[0].children[0] == "Session not found"
print(f"Alert OK: {err.attrs['class']}")

Alert OK: alert alert-error


## Empty State

Placeholder shown when there are zero sessions in the managed flow. Uses a large faded `inbox` icon and two lines of copy.

In [ ]:
#| export
def render_empty_state(
    message:str="No sessions found.", # Primary message
    detail:str="Start a new session to begin a workflow.", # Secondary detail
) -> Any: # Empty state element
    """Render an empty state placeholder with an icon and two lines of copy."""
    return Div(
        lucide_icon("inbox", size=icons.empty_state, cls=text_tiers.subtle),
        P(message, cls=combine_classes(
            font_size.lg, font_weight.medium, m.t(4)
        )),
        P(detail, cls=combine_classes(
            font_size.sm, text_tiers.tertiary
        )),
        cls=combine_classes(
            flex_display, flex_direction.col, items.center, justify.center,
            text_align.center, p(12)
        )
    )

In [ ]:
empty = render_empty_state()
assert empty.tag == "div"
# Icon first, then two P's.
assert empty.children[0].tag == "svg"
assert empty.children[1].tag == "p"
assert empty.children[1].children[0] == "No sessions found."
assert empty.children[2].tag == "p"
assert empty.children[2].children[0] == "Start a new session to begin a workflow."

# Custom copy.
empty2 = render_empty_state(message="Nothing here yet", detail="Create your first session")
assert empty2.children[1].children[0] == "Nothing here yet"
print("Empty state OK")

Empty state OK


## Active Session Badge

Visual indicator applied to the currently active session row — the session whose ID matches `sess["_workflow_session_id"]`. Uses a `circle-dot` lucide icon in a small success-colored badge.

In [ ]:
#| export
def render_active_session_badge(
    label:str="Active", # Badge label text
    badge_id:str="", # Optional HTML ID for the badge
) -> Any: # Span element containing the badge
    """Render the "Active" indicator badge for the currently active session row."""
    kwargs = {
        "cls": combine_classes(
            badge, badge_colors.success, badge_sizes.sm,
            flex_display, items.center, gap(1)
        )
    }
    if badge_id: kwargs["id"] = badge_id
    return Span(
        lucide_icon("circle-dot", size=icons.dense_inline),
        label,
        **kwargs
    )

In [ ]:
active = render_active_session_badge(badge_id="wsm-active-badge")
assert active.tag == "span"
assert active.attrs["id"] == "wsm-active-badge"
assert "badge" in active.attrs["class"]
assert "badge-success" in active.attrs["class"]
assert "badge-sm" in active.attrs["class"]
# Icon then label.
assert active.children[0].tag == "svg"
assert active.children[1] == "Active"

# Custom label.
resume = render_active_session_badge(label="Current")
assert resume.children[1] == "Current"
print(f"Active badge classes: {active.attrs['class']}")

Active badge classes: badge badge-success badge-sm flex items-center gap-1


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()